# CUSP real-time benchmark on Kaggle (GPU T4)
Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**.
Add two private datasets with **+ Add Input**:
1. `cusp-code`: the contents of `cusp_code.zip` (including `kaggle_bench.py`)
2. `kitti-tracking`: the KITTI Tracking `training` folder with `image_02/`, `label_02/`, `calib/`, `oxts/` (sequences 0000-0020)

Edit the two paths in the next cell if your dataset names differ.

In [ ]:
import glob, os, shutil, subprocess
KITTI = glob.glob('/kaggle/input/**/label_02', recursive=True)[0].rsplit('/label_02', 1)[0]
CODE  = os.path.dirname(glob.glob('/kaggle/input/**/cusp_eval.py', recursive=True)[0])
print('KITTI:', KITTI); print('CODE :', CODE)
os.chdir('/kaggle/working')
for f in glob.glob(CODE + '/*.py'): shutil.copy(f, '.')
!pip install -q -U ultralytics lap
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

## 1. Export TensorRT engines (FP16 and INT8). Takes about 5-15 min.

In [ ]:
from ultralytics import YOLO
def export(src, dst, **kw):
    if os.path.exists(dst): print('exists', dst); return
    p = YOLO(src).export(format='engine', imgsz=640, device=0, **kw)
    os.rename(p, dst); print('->', dst)
export('yolo26s.pt', 'yolo26s_fp16.engine', half=True)
export('yolo26s.pt', 'yolo26s_int8.engine', int8=True, data='coco128.yaml')   # INT8 calibration images
export('yolo26n.pt', 'yolo26n_fp16.engine', half=True)

## 2. Per-stage latency on the KITTI test sequences 0011-0020

In [ ]:
!python kaggle_bench.py --root "$KITTI" --seqs 0011-0020 --device 0 \
    --models yolo26s.pt yolo26s_fp16.engine yolo26s_int8.engine yolo26n_fp16.engine --out bench

## 3. Accuracy and CUSP timing per model (coverage, unsafe speed caps, latency)

In [ ]:
for w in ['yolo26s.pt', 'yolo26s_fp16.engine', 'yolo26s_int8.engine', 'yolo26n_fp16.engine']:
    out = 'out_' + w.split('.')[0]
    !python cusp_eval.py --calib kitti=$KITTI:0-10 --test kitti=$KITTI:11-20 --cam-height 1.65 \
        --frontend yolo --weights $w --device 0 --out $out

## 4. Package results for download (Output tab, or the link below)

In [ ]:
!zip -qr kaggle_results.zip bench out_yolo26* -x '*/cache/*'
from IPython.display import FileLink; FileLink('kaggle_results.zip')